# 🧠 Brain-Inspired AI Demo

This notebook demonstrates the key features of the Brain-Inspired AI system.

本notebook演示类脑人工智能系统的关键特性。

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Import our modules
from src.models.model_integration import BrainInspiredModel
from src.core.neurons import LIFNeuron, SpikingLayer
from src.core.stdp import STDPLearner
from src.memory.hippocampus import HippocampalMemory
from src.tools.web_tools import ToolRegistry

print("✓ Imports successful!")

## 2. Initialize the Model

Load the brain-inspired AI model with all components.

In [ ]:
# Initialize model
print("Loading Brain-Inspired AI model...")

model = BrainInspiredModel(
    base_model_name="Qwen/Qwen2.5-0.5B-Instruct",
    device="auto",
    use_snn=True,
    use_memory=True,
)

print("✓ Model loaded!")
print(f"Device: {model.device}")
print(f"Hidden size: {model.hidden_size}")

## 3. Text Generation

Test basic text generation capabilities.

In [ ]:
# Simple generation
prompt = "Explain the concept of neural networks in simple terms:"

print("Prompt:", prompt)
print("\nResponse:")

response = model.generate(
    prompt=prompt,
    max_new_tokens=100,
    temperature=0.7,
)

print(response)

## 4. Streaming Generation

Demonstrate high-refresh-rate streaming generation.

In [ ]:
# Streaming generation
prompt = "What are the benefits of exercise?"

print("Prompt:", prompt)
print("\nStreaming Response:")
print("-" * 50)

for token in model.generate_stream(
    prompt=prompt,
    max_new_tokens=50,
    temperature=0.7,
):
    if not token.is_special:
        print(token.token, end="", flush=True)

print("\n" + "-" * 50)

## 5. Memory System

Test the hippocampal memory system.

In [ ]:
# Store some memories
memories = [
    "The capital of France is Paris.",
    "Machine learning is a subset of AI.",
    "Python is a popular programming language.",
    "Neural networks are inspired by the brain.",
]

print("Storing memories...")
for mem in memories:
    embedding = model.encode_multimodal(text=mem)
    model.store_memory(mem, embedding, memory_type="semantic")
    print(f"  ✓ Stored: {mem[:50]}...")

print(f"\nTotal memories stored: {len(model.hippocampus.engrams)}")

In [ ]:
# Retrieve memories
query = "What is machine learning?"
print(f"Query: {query}\n")

query_embedding = model.encode_multimodal(text=query)
retrieved = model.retrieve_memory(query_embedding, top_k=3)

print("Retrieved memories:")
for i, (mem, score) in enumerate(retrieved, 1):
    content = mem.raw_content if hasattr(mem, 'raw_content') else str(mem)
    print(f"{i}. [{score:.3f}] {content}")

## 6. Spiking Neural Network

Demonstrate SNN components.

In [ ]:
# Create LIF neuron
lif = LIFNeuron(
    tau_mem=20.0,
    v_thresh=1.0,
    spike_surrogate="fast_sigmoid",
)

# Generate input current
time_steps = 100
input_current = torch.randn(1, time_steps, 10) * 0.5 + 0.5

# Process through LIF
spikes = []
state = None

for t in range(time_steps):
    spike, state = lif(input_current[:, t, :], state)
    spikes.append(spike)

spikes = torch.stack(spikes, dim=1)

# Visualize
plt.figure(figsize=(12, 4))
plt.imshow(spikes[0].T.numpy(), aspect='auto', cmap='binary')
plt.xlabel('Time Step')
plt.ylabel('Neuron')
plt.title('LIF Neuron Spike Raster')
plt.colorbar(label='Spike')
plt.show()

print(f"Spike rate: {spikes.mean().item():.3f}")

## 7. STDP Learning

Demonstrate spike-timing-dependent plasticity.

In [ ]:
# Create STDP learner
stdp = STDPLearner(
    A_plus=0.01,
    A_minus=0.01,
    tau_plus=20.0,
    tau_minus=20.0,
)

# Generate spike trains
pre_spikes = torch.zeros(1, 50, 10)
post_spikes = torch.zeros(1, 50, 10)

# Create correlated spikes (pre before post)
for t in range(10, 40):
    pre_spikes[0, t, :] = (torch.rand(10) > 0.7).float()
    post_spikes[0, t+2, :] = (torch.rand(10) > 0.7).float()

# Initial weights
weights = torch.randn(10, 10) * 0.1
initial_weights = weights.clone()

# Apply STDP
delta_w = stdp(weights, pre_spikes, post_spikes)

# Visualize weight changes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(initial_weights.numpy(), cmap='RdBu_r')
axes[0].set_title('Initial Weights')
axes[0].set_xlabel('Pre Neuron')
axes[0].set_ylabel('Post Neuron')

axes[1].imshow(delta_w.numpy(), cmap='RdBu_r')
axes[1].set_title('Weight Change (Δw)')
axes[1].set_xlabel('Pre Neuron')
axes[1].set_ylabel('Post Neuron')

axes[2].imshow((initial_weights + delta_w).numpy(), cmap='RdBu_r')
axes[2].set_title('Updated Weights')
axes[2].set_xlabel('Pre Neuron')
axes[2].set_ylabel('Post Neuron')

plt.tight_layout()
plt.show()

print(f"Mean weight change: {delta_w.mean().item():.6f}")

## 8. Tool Usage

Test tool integration.

In [ ]:
# Initialize tool registry
tools = ToolRegistry()

# List available tools
print("Available tools:")
for tool in tools.get_available_tools():
    print(f"  - {tool['name']}: {tool['description']}")

In [ ]:
# Use calculator
result = tools.call("calculate", expression="15 * 17 + 23")
print(f"Calculator result: {result.result}")

# Use Wikipedia search
result = tools.call("wikipedia_search", query="artificial intelligence", sentences=2)
if result.success:
    print(f"\nWikipedia results:")
    for item in result.result[:2]:
        print(f"  - {item['title']}: {item['summary'][:100]}...")
else:
    print(f"Error: {result.error}")

## 9. Benchmark

Run quick benchmark.

In [ ]:
from tests.benchmark import SpeedBenchmark

speed_benchmark = SpeedBenchmark(model)
result = speed_benchmark.evaluate(verbose=True)

print(f"\nSpeed Benchmark Results:")
print(f"  Tokens per second: {result.metrics['tokens_per_second']:.2f}")
print(f"  Average generation time: {result.metrics['avg_generation_time']:.2f}s")

## 10. Save and Load

Demonstrate model persistence.

In [ ]:
import os

# Save model
save_path = "../checkpoints/demo_model"
os.makedirs(save_path, exist_ok=True)

print(f"Saving model to {save_path}...")
model.save(save_path)
print("✓ Model saved!")

# List saved files
print("\nSaved files:")
for f in os.listdir(save_path):
    print(f"  - {f}")

## Summary

This notebook demonstrated:
- ✅ Model initialization
- ✅ Text generation (streaming and non-streaming)
- ✅ Hippocampal memory system
- ✅ Spiking neural networks
- ✅ STDP learning
- ✅ Tool integration
- ✅ Benchmarking
- ✅ Model persistence

For more details, see the full documentation.